# Detecting Missing Values in Pandas DataFrames

## Learning Objectives
- Understand what missing values represent
- Detect missing values in a DataFrame
- Identify missing values at row and column level
- Interpret missing-value summaries correctly
- Build habits of checking for missing data early

---

## Why Detecting Missing Values Matters

**Missing data is one of the most common data quality issues in real-world datasets.**

Common issues caused by ignoring missing values:
- Incorrect statistical calculations (means, medians, counts)
- Silent analysis errors
- Misleading visualizations
- Invalid model training
- Poor business decisions

**Detection comes before cleaning. You must identify the problem before you can solve it.**

Think of missing value detection as a **data health check** that should happen early and often.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np

print("✓ Libraries imported successfully!")

---

## Part 1: Understanding Missing Values

### What Are Missing Values?

Missing values represent **the absence of data** in a dataset. They can occur for many reasons:

- **Not collected**: Survey respondent skipped a question
- **Lost**: Data corruption or transfer errors
- **Not applicable**: Field doesn't apply to this record
- **Not yet available**: Data will be added later
- **Measurement failure**: Sensor malfunction

### How Pandas Represents Missing Values

Pandas uses several special values to represent missing data:

1. **`NaN`** (Not a Number) - Most common, used for numeric data
2. **`None`** - Python's null object, used for object dtype columns
3. **`pd.NA`** - Newer scalar missing value indicator
4. **`pd.NaT`** - Missing value for datetime data

**Important**: Empty strings `""` and zero `0` are **NOT** missing values! They are actual values.

In [ ]:
# Let's see what missing values look like
print("Examples of missing value representations:")
print("=" * 60)

# Create a small example DataFrame with different missing value types
example_data = {
    'employee_id': [101, 102, 103, 104, 105],
    'score': [85, np.nan, 92, 88, np.nan],  # NaN in numeric column
    'department': ['IT', None, 'HR', 'Sales', 'IT'],  # None in object column
    'hire_date': pd.to_datetime(['2024-01-15', pd.NaT, '2024-03-10', '2024-02-20', pd.NaT]),  # NaT in datetime
}

example_df = pd.DataFrame(example_data)

print("\nExample DataFrame with missing values:")
print(example_df)

print("\n" + "=" * 60)
print("Notice:")
print("- Numeric column shows NaN")
print("- Object column shows None (displayed as NaN)")
print("- Datetime column shows NaT (Not a Time)")

---

## Part 2: Load Real Dataset with Missing Values

Let's load the employee survey dataset that contains missing values.

In [ ]:
# Load the employee survey data with missing values
df = pd.read_csv('../data/raw/employee_survey_with_missing_2026_Q1.csv')

print("✓ Data loaded successfully!")
print(f"\nDataset shape: {df.shape}")
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")

In [ ]:
# Preview the first few rows
print("First 10 rows of the dataset:")
print("=" * 100)
df.head(10)

In [ ]:
# Get basic information about the dataset
print("Dataset Information:")
print("=" * 100)
df.info()

---

## Part 3: Detecting Missing Values

### Primary Methods for Detection

Pandas provides two main methods to detect missing values:

1. **`isna()`** - Returns True where values are missing (recommended)
2. **`isnull()`** - Alias for `isna()` (same functionality)

Both methods return a **boolean DataFrame** of the same shape as the original.

### Method 1: isna() - Element-wise Detection

In [ ]:
# Use isna() to create a boolean mask
missing_mask = df.isna()

print("Boolean mask showing missing values (True = missing):")
print("=" * 100)
print("\nFirst 10 rows:")
print(missing_mask.head(10))

print("\n" + "=" * 100)
print("\nKey observation:")
print("- True indicates a missing value")
print("- False indicates a present value")
print("- The result has the same shape as the original DataFrame")

### Method 2: notna() - Detecting Present Values

Sometimes you want to identify **non-missing** values instead:

- **`notna()`** - Returns True where values are present
- **`notnull()`** - Alias for `notna()`

This is useful for filtering rows or counting valid entries.

In [ ]:
# Use notna() to find present values
present_mask = df.notna()

print("Boolean mask showing present values (True = present):")
print("=" * 100)
print("\nFirst 10 rows:")
print(present_mask.head(10))

print("\n" + "=" * 100)
print("Relationship between isna() and notna():")
print("- notna() is the logical opposite of isna()")
print("- isna() + notna() = complete dataset coverage")

---

## Part 4: Counting Missing Values

### Count Missing Values Per Column

The most common task is summarizing missing values by column.

**Technique**: Combine `isna()` with `sum()`

Why this works:
- `isna()` returns True/False
- `sum()` treats True as 1 and False as 0
- Result is the count of missing values

In [ ]:
# Count missing values per column
missing_counts = df.isna().sum()

print("Missing Values Per Column:")
print("=" * 60)
print(missing_counts)

print("\n" + "=" * 60)
print(f"Total cells in dataset: {df.shape[0] * df.shape[1]}")
print(f"Total missing cells: {df.isna().sum().sum()}")

### Calculate Missing Value Percentages

**Percentages provide better context than raw counts.**

A column with 5 missing values out of 10 (50%) is very different from 5 out of 10,000 (0.05%).

In [ ]:
# Calculate percentage of missing values per column
missing_percentage = (df.isna().sum() / len(df)) * 100

print("Percentage of Missing Values Per Column:")
print("=" * 60)
print(missing_percentage.round(2))

print("\n" + "=" * 60)
print("Interpretation Guide:")
print("- 0%: No missing values (complete column)")
print("- 1-5%: Minor missingness")
print("- 5-20%: Moderate missingness")
print("- 20-50%: Significant missingness")
print("- >50%: Severe missingness (consider dropping column)")

### Create a Comprehensive Missing Value Summary

Combine counts and percentages for a complete overview.

In [ ]:
# Create a summary DataFrame
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isna().sum().values,
    'Missing_Percentage': ((df.isna().sum() / len(df)) * 100).values.round(2),
    'Data_Type': df.dtypes.values,
    'Non_Missing_Count': df.notna().sum().values
})

# Sort by missing count (descending) to see worst columns first
missing_summary = missing_summary.sort_values('Missing_Count', ascending=False)

print("Comprehensive Missing Value Summary:")
print("=" * 100)
print(missing_summary.to_string(index=False))

print("\n" + "=" * 100)
print("\nColumns with NO missing values:")
complete_columns = missing_summary[missing_summary['Missing_Count'] == 0]['Column'].tolist()
print(complete_columns if complete_columns else "None - all columns have missing values")

print("\nColumns with missing values:")
incomplete_columns = missing_summary[missing_summary['Missing_Count'] > 0]['Column'].tolist()
print(f"{len(incomplete_columns)} out of {len(df.columns)} columns have missing data")

---

## Part 5: Inspecting Rows with Missing Data

### Find Rows with ANY Missing Value

Often you want to see the actual rows that contain missing data.

**Technique**: Use `isna().any(axis=1)` to check if any column in a row is missing.

In [ ]:
# Identify rows with at least one missing value
rows_with_missing = df[df.isna().any(axis=1)]

print(f"Total rows with at least one missing value: {len(rows_with_missing)}")
print(f"Total rows in dataset: {len(df)}")
print(f"Percentage of rows affected: {(len(rows_with_missing) / len(df) * 100):.2f}%")

print("\n" + "=" * 100)
print("\nFirst 10 rows with missing data:")
print("=" * 100)
rows_with_missing.head(10)

### Find Rows with NO Missing Values (Complete Cases)

Complete cases are rows where every column has a value.

In [ ]:
# Identify complete rows (no missing values)
complete_rows = df[df.notna().all(axis=1)]

print(f"Total complete rows (no missing values): {len(complete_rows)}")
print(f"Total rows in dataset: {len(df)}")
print(f"Percentage of complete rows: {(len(complete_rows) / len(df) * 100):.2f}%")

print("\n" + "=" * 100)
print("\nFirst 10 complete rows:")
print("=" * 100)
complete_rows.head(10)

### Find Rows Missing Specific Columns

Sometimes you need to focus on missing values in critical columns.

In [ ]:
# Find rows where 'satisfaction_score' is missing (a critical column)
missing_satisfaction = df[df['satisfaction_score'].isna()]

print("Rows with missing satisfaction_score:")
print("=" * 100)
print(f"Count: {len(missing_satisfaction)}")
print("\nThese rows:")
print(missing_satisfaction[['employee_id', 'department', 'satisfaction_score', 'response_text']])

print("\n" + "=" * 100)

# Find rows where department is missing
missing_department = df[df['department'].isna()]

print("\nRows with missing department:")
print("=" * 100)
print(f"Count: {len(missing_department)}")
print("\nThese rows:")
print(missing_department[['employee_id', 'department', 'satisfaction_score']])

### Find Rows with Multiple Missing Values

Rows with many missing values might need special handling.

In [ ]:
# Count missing values per row
missing_per_row = df.isna().sum(axis=1)

print("Distribution of missing values across rows:")
print("=" * 60)
print(missing_per_row.value_counts().sort_index())

print("\n" + "=" * 100)

# Find rows with 3 or more missing values
severely_incomplete = df[missing_per_row >= 3]

print(f"\nRows with 3 or more missing values: {len(severely_incomplete)}")
if len(severely_incomplete) > 0:
    print("\nThese rows:")
    print(severely_incomplete)

---

## Part 6: Patterns in Missing Data

### Analyze Missing Data Patterns

Understanding **why** data is missing can guide your cleaning strategy.

In [ ]:
# Check if certain columns tend to be missing together
print("Columns that are often missing together:")
print("=" * 100)

# Create a correlation matrix of missing values
missing_correlation = df.isna().corr()

print("\nCorrelation of missing values between columns:")
print("(1.0 = always missing together, 0.0 = no relationship)")
print("\n", missing_correlation.round(2))

In [ ]:
# Analyze missing values by department
print("Missing value analysis by department:")
print("=" * 100)

# For each department, count missing satisfaction scores
dept_missing = df.groupby('department')['satisfaction_score'].apply(lambda x: x.isna().sum())

print("\nMissing satisfaction_score by department:")
print(dept_missing)

print("\n" + "=" * 100)
print("\nInterpretation:")
print("- Are certain departments less likely to respond?")
print("- Could this introduce bias in analysis?")
print("- Should we investigate why some departments have more missing data?")

---

## Part 7: Quick Detection Functions

### Create Reusable Detection Functions

Build helper functions for quick missing value checks.

In [ ]:
def missing_value_report(dataframe):
    """
    Generate a comprehensive missing value report for a DataFrame.
    
    Parameters:
    -----------
    dataframe : pd.DataFrame
        The DataFrame to analyze
    
    Returns:
    --------
    pd.DataFrame
        Summary of missing values
    """
    total_cells = dataframe.shape[0] * dataframe.shape[1]
    total_missing = dataframe.isna().sum().sum()
    
    print("="*80)
    print("MISSING VALUE REPORT")
    print("="*80)
    print(f"\nDataset Shape: {dataframe.shape[0]} rows × {dataframe.shape[1]} columns")
    print(f"Total Cells: {total_cells:,}")
    print(f"Missing Cells: {total_missing:,}")
    print(f"Missing Percentage: {(total_missing/total_cells*100):.2f}%")
    
    print("\n" + "="*80)
    print("COLUMN-LEVEL SUMMARY")
    print("="*80)
    
    summary = pd.DataFrame({
        'Column': dataframe.columns,
        'Missing': dataframe.isna().sum().values,
        'Percent': ((dataframe.isna().sum() / len(dataframe)) * 100).values.round(2),
        'Type': dataframe.dtypes.values
    })
    
    summary = summary[summary['Missing'] > 0].sort_values('Missing', ascending=False)
    
    if len(summary) == 0:
        print("\n✓ No missing values detected in any column!")
    else:
        print(f"\n{len(summary)} columns with missing values:\n")
        print(summary.to_string(index=False))
    
    print("\n" + "="*80)
    print("ROW-LEVEL SUMMARY")
    print("="*80)
    
    rows_with_missing = dataframe[dataframe.isna().any(axis=1)]
    print(f"\nRows with missing values: {len(rows_with_missing)} ({len(rows_with_missing)/len(dataframe)*100:.2f}%)")
    print(f"Complete rows: {len(dataframe) - len(rows_with_missing)} ({(len(dataframe)-len(rows_with_missing))/len(dataframe)*100:.2f}%)")
    
    print("\n" + "="*80)
    
    return summary

# Test the function
summary = missing_value_report(df)

---

## Part 8: Best Practices for Missing Value Detection

### Key Takeaways

1. **Always check for missing values early** - Right after loading data
2. **Understand the pattern** - Why is data missing?
3. **Document your findings** - Record what you discover
4. **Never ignore missing values** - They affect all downstream analysis
5. **Detection before action** - Detect first, then decide how to handle

### Common Detection Workflow

```python
# Step 1: Load data
df = pd.read_csv('data.csv')

# Step 2: Quick check
print(df.isna().sum())

# Step 3: Detailed analysis
missing_summary = missing_value_report(df)

# Step 4: Investigate patterns
# Look at rows with missing values
# Analyze if certain groups have more missing data

# Step 5: Decide on handling strategy
# (This comes AFTER detection)
```

### What NOT to Do

❌ **Don't** start analysis without checking for missing values  
❌ **Don't** assume data is complete  
❌ **Don't** immediately drop missing values without understanding why they're missing  
❌ **Don't** fill missing values without proper justification  
❌ **Don't** ignore the impact of missing values on your results  

### What TO Do

✅ **Do** check for missing values immediately after loading data  
✅ **Do** understand percentages, not just counts  
✅ **Do** investigate patterns in missingness  
✅ **Do** document your findings  
✅ **Do** consider the impact on your analysis  

---

## Practice Exercises

Try these exercises to reinforce your learning:

In [ ]:
# Exercise 1: Find the column with the highest percentage of missing values
print("Exercise 1: Column with most missing values")
print("="*60)

# Your code here
missing_pct = (df.isna().sum() / len(df) * 100)
worst_column = missing_pct.idxmax()
worst_pct = missing_pct.max()

print(f"Column: {worst_column}")
print(f"Missing percentage: {worst_pct:.2f}%")

In [ ]:
# Exercise 2: How many rows have exactly 1 missing value?
print("Exercise 2: Rows with exactly 1 missing value")
print("="*60)

# Your code here
missing_count_per_row = df.isna().sum(axis=1)
rows_with_one_missing = (missing_count_per_row == 1).sum()

print(f"Rows with exactly 1 missing value: {rows_with_one_missing}")

In [ ]:
# Exercise 3: Find rows where both 'satisfaction_score' AND 'work_life_balance' are missing
print("Exercise 3: Rows missing both satisfaction_score and work_life_balance")
print("="*60)

# Your code here
both_missing = df[df['satisfaction_score'].isna() & df['work_life_balance'].isna()]

print(f"Number of rows: {len(both_missing)}")
print("\nThese rows:")
print(both_missing[['employee_id', 'satisfaction_score', 'work_life_balance']])

---

## Summary

### What You Learned

1. **Understanding Missing Values**
   - What missing values represent
   - How Pandas represents them (NaN, None, NaT)
   - Why they occur in real datasets

2. **Detection Methods**
   - `isna()` / `isnull()` for detecting missing values
   - `notna()` / `notnull()` for detecting present values
   - Boolean masking for identification

3. **Counting and Summarizing**
   - Count missing values per column: `df.isna().sum()`
   - Calculate percentages for context
   - Create comprehensive summaries

4. **Row-Level Inspection**
   - Find rows with any missing values: `df[df.isna().any(axis=1)]`
   - Find complete rows: `df[df.notna().all(axis=1)]`
   - Identify rows with multiple missing values

5. **Pattern Analysis**
   - Detect correlations in missingness
   - Analyze missing data by groups
   - Understand systematic patterns

### Next Steps

Now that you can **detect** missing values, the next steps are:

1. **Handling strategies** - Decide whether to:
   - Drop missing values
   - Impute (fill) missing values
   - Keep them with special handling

2. **Impact analysis** - Understand how missing values affect:
   - Statistical calculations
   - Visualizations
   - Machine learning models

3. **Documentation** - Always document:
   - What missing values were detected
   - Why they might be missing
   - How you decided to handle them

### Remember

**Detection is the foundation.** You cannot properly handle missing values until you know:
- Where they are
- How many there are
- Why they might be missing

Make missing value detection a **habit** at the start of every data analysis project.

---

## 🎥 Video Demonstration Task

Record a 2-minute screen capture video demonstrating:

1. **Load the dataset** (5-10 seconds)
2. **Detect missing values** using `isna()` (15-20 seconds)
3. **Count missing values per column** (20-30 seconds)
4. **Identify rows with missing data** (20-30 seconds)
5. **Explain why detection matters** (30-40 seconds)

Tips for your video:
- Keep it concise and focused
- Explain what you're doing as you code
- Show the output clearly
- Emphasize the importance of early detection

---

**Congratulations!** You now have a solid foundation in detecting missing values in Pandas DataFrames. This is a critical skill for data quality and analysis.